In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import pandas_gbq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [ ]:
gcp_service_acc_file_aluno = "../secrets/gcp-sa-seminarios.json"
gcp_id_do_projeto = "meu-projeto-eng-dados"
my_gcp_cred = service_account.Credentials.from_service_account_file(gcp_service_acc_file_aluno)
my_gcp_client = bigquery.Client(credentials=my_gcp_cred)

In [3]:
df_silver = my_gcp_client.query(
    f"""
        SELECT 
            * 
        FROM
            silver.complete_dc_marvel
    """,
).to_dataframe(create_bqstorage_client=False)

In [16]:
df_silver.head()

,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BREAK_EVEN,BUDGET_x,CHARACTER_FAMILY,COUNTRY_ORIGIN,DIRECTOR,DISTRIBUTOR,DOMESTIC,FILM,FILMING_LOCATION,FRANCHISE,GENRE,GROSS_OPENING_WEEKEND,GROSS_TO_BUDGET,GROSS_US_CANADA,GROSS_WORLD_WIDE,ID,INFLATION_ADJUSTED_BUDGET,INFLATION_ADJUSTED_WORLDWIDE_GROSS,LANGUAGE,LINK,MALE_FEMALE_LED,MCU,MINUTES,NOMINATION,OSCAR,PHASE,PRODUCTION_COMPANY,RATING_IMDB,RATING_MPA,ROTTEN_TOMATOES_CRITIC_SCORE,STAR,US_RELEASE_DATE,VOTE,WIN,WRITER,YEAR
0,121200000.0,Flop,84500000.0,Birds of Prey,NaN,NaN,Warner Bros.,0.41,Birds of Prey (and the Fantabulous Emancipatio...,NaN,DC,NaN,NaN,2.43,NaN,NaN,NaN,99482958.0,2.417881e+08,NaN,NaN,Female,False,109.0,0,0,NaN,NaN,NaN,NaN,78,NaN,2020-02-07,0,0,NaN,2020
1,186597000.0,Flop,200000000.0,X-Men,NaN,NaN,Twentieth Century Fox,0.26,Dark Phoenix,NaN,Marvel,NaN,NaN,1.26,NaN,NaN,NaN,200000000.0,2.524430e+08,NaN,NaN,Female,False,113.0,0,0,NaN,NaN,NaN,NaN,22,NaN,2019-06-07,0,0,NaN,2019
2,741395911.0,Success,160000000.0,Spider-Man,NaN,NaN,Sony Pictures Entertainment,0.35,Spider-Man: Far From Home,NaN,Marvel,NaN,NaN,7.07,NaN,NaN,NaN,160000000.0,1.131928e+09,NaN,NaN,Male,True,129.0,0,0,3.0,NaN,NaN,NaN,90,NaN,2019-07-02,0,0,NaN,2019
3,22600000.0,Success,10000000.0,Teen Titans,NaN,NaN,Warner Bros.,0.57,Teen Titans Go! To the Movies,NaN,DC,NaN,NaN,5.24,NaN,NaN,NaN,12134333.0,6.357205e+07,NaN,NaN,Co-starring,False,84.0,0,0,NaN,NaN,NaN,NaN,91,NaN,2018-07-27,0,0,NaN,2018
4,636918053.0,Success,250000000.0,Batman,NaN,NaN,Warner Bros.,0.41,The Dark Knight Rises,NaN,DC,NaN,NaN,4.34,NaN,NaN,tt1345836,331783042.0,1.440028e+09,NaN,https://www.imdb.com/title/tt1345836,Male,False,164.0,0,0,NaN,NaN,8.4,PG-13,87,NaN,2012-07-20,19000000,0,NaN,2012


In [22]:
dim_film = (
    df_silver[
        [
            "FILM",
            "GENRE",
            "RATING_MPA",
            "LANGUAGE",
            "BREAK_EVEN",
            "MCU",
            "MALE_FEMALE_LED",
            "PHASE",
        ]
    ]
    .drop_duplicates()
    .sort_values(by="FILM")
    .reset_index(drop=True)
)

dim_film["ID_FILM"] = dim_film.index +1

In [33]:
dim_staff = (
    df_silver[["DIRECTOR", "WRITER", "STAR"]]
    .drop_duplicates()
    .sort_values(by=["DIRECTOR", "WRITER"])
    .reset_index(drop=True)
)

dim_staff["ID_STAFF"] = dim_staff.index + 1

dim_staff.head(2)

,DIRECTOR,WRITER,STAR,ID_STAFF
0,Alan Taylor,Christopher L Yost,"Chris Hemsworth, Natalie Portman, Tom Hiddleston",1
1,Andy Muschietti,Christina Hodson,"Ezra Miller, Michael Keaton, Sasha Calle",2


In [40]:
dim_franchise = (
    df_silver[["FRANCHISE", "CHARACTER_FAMILY"]]
    .drop_duplicates()
    .sort_values(by=["FRANCHISE", "CHARACTER_FAMILY"])
    .reset_index(drop=True)
)

dim_franchise["ID_FRANCHISE"] = dim_franchise.index + 1

dim_franchise.head(5)

,FRANCHISE,CHARACTER_FAMILY,ID_FRANCHISE
0,DC,Aquaman,1
1,DC,Batman,2
2,DC,Birds of Prey,3
3,DC,Blue Beetle,4
4,DC,Constantine,5


In [43]:
cols_prod = ["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"]

dim_production = (
    df_silver[cols_prod]
    .drop_duplicates()
    .sort_values(by=cols_prod)
    .reset_index(drop=True)
)

dim_production["ID_PRODUCTION"] = dim_production.index + 1

dim_production.head(5)

,PRODUCTION_COMPANY,DISTRIBUTOR,COUNTRY_ORIGIN,FILMING_LOCATION,ID_PRODUCTION
0,20th Century Studios,20th Century Studios,United States,"Medfield State Hospital 45 Hospital Road, Med...",1
1,Amen Ra Films,New Line Cinema,United States,"Vancouver, British Columbia, Canada",2
2,Animal Logic,Warner Bros.,United States,"Disney Studios, Moore Park, Sydney, New South ...",3
3,Artistry Limited,Tri-Star Pictures,United Kingdom,"Chicago, Illinois, USA",4
4,Atlas Entertainment,Warner Bros.,United States,"Fuerteventura, Canary Islands, Spain",5


In [ ]:
dim_date = df_silver[["US_RELEASE_DATE"]].dropna().drop_duplicates().sort_values(by="US_RELEASE_DATE").reset_index(drop=True)

In [47]:
dim_date = df_silver[["US_RELEASE_DATE"]].dropna().drop_duplicates().sort_values(by="US_RELEASE_DATE").reset_index(drop=True)

dim_date["YEAR"] = dim_date["US_RELEASE_DATE"].dt.year
dim_date["MONTH"] = dim_date["US_RELEASE_DATE"].dt.month
dim_date["QUARTER"] = dim_date["US_RELEASE_DATE"].dt.quarter

dim_date["ID_DATE"] = dim_date["US_RELEASE_DATE"].dt.strftime("%Y%m%d").astype(int)

dim_date.head(5)

,US_RELEASE_DATE,YEAR,MONTH,QUARTER,ID_DATE
0,1978-12-15,1978,12,4,19781215
1,1981-06-19,1981,6,2,19810619
2,1983-06-17,1983,6,2,19830617
3,1984-11-21,1984,11,4,19841121
4,1986-08-01,1986,8,3,19860801


In [52]:
fact_movies = df_silver.copy()

In [53]:
fact_movies = fact_movies.merge(
    dim_film,
    on=["FILM", "GENRE", "RATING_MPA", "LANGUAGE", "BREAK_EVEN", "MCU", "MALE_FEMALE_LED", "PHASE"],
    how="left",
)

In [54]:
fact_movies = fact_movies.merge(
    dim_staff,
    on=["DIRECTOR", "WRITER", "STAR"],
    how="left",
)

In [55]:
fact_movies = fact_movies.merge(
    dim_franchise,
    on=["FRANCHISE", "CHARACTER_FAMILY"],
    how="left",
)

In [56]:
fact_movies = fact_movies.merge(
    dim_production,
    on=["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"],
    how="left",
)

In [57]:
fact_movies = fact_movies.merge(
    dim_date,
    on=["US_RELEASE_DATE"],
    how="left",
)

In [59]:
fact_movies = fact_movies[
    [
        "ID_FILM", "ID_STAFF", "ID_FRANCHISE", "ID_PRODUCTION",
        "ID_DATE", "BUDGET_x", "INFLATION_ADJUSTED_BUDGET",
        "GROSS_WORLD_WIDE", "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
        "GROSS_US_CANADA", "DOMESTIC", "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
        "GROSS_OPENING_WEEKEND", "GROSS_TO_BUDGET",
        "RATING_IMDB", "ROTTEN_TOMATOES_CRITIC_SCORE",
        "VOTE", "OSCAR", "NOMINATION", "WIN", "MINUTES"
    ]
]

In [61]:
minhas_tabelas = {
    "dim_date": dim_date,
    "dim_film": dim_film,
    "dim_franchise": dim_franchise,
    "dim_production": dim_production,
    "dim_staff": dim_staff,
    "fact_movies": fact_movies    
}

In [64]:
for key, value in minhas_tabelas.items():
    print(key)
    pandas_gbq.to_gbq(
        value,
        project_id=gcp_id_do_projeto,
        destination_table=f"gold.{key}",
        if_exists="replace",
        credentials=my_gcp_cred,
    )

dim_date
dim_film
dim_franchise
dim_production
dim_staff
fact_movies
